In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,RobustScaler
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping

df = pd.read_csv('training_data_ready.csv')
df = df.drop([21,43,92,159], axis=0) #[21,43,92,159] # [92,99,31,169,150,43] #[17,21,43,53,62,92,129,150,159,166,169] #[15,17,21,30,35,43,53,62,67,72,73,75,91,92,129,139,142,150,159,166,169]
target_cols = ['x', 'y', 'z','Gewicht'] 
x_cols = [col for col in df.columns if col not in target_cols + ['source_file','Objektname','Objektkategorie','OAufnahme','Schwierigkeit','Gewicht']]
    
X = df[x_cols]
y = df[target_cols]
    # 3. Train-Test-Split (80% Training, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



In [2]:
#---------------------------- Ausgaben Beste Modelle: -----------------------
#Platz  | Scaler    | Netz-Struktur      | L2     | Dropout | LR     | Val-Loss (MSE) | Val-MAE 
#-----------------------------------------------------------------------------------------------
#1    | robust    | [128, 64]          | 0.1    | 0.0     | 0.05   | 314.078888     | 12.3596 
#2    | robust    | [512, 256, 128, 64] | 0.5    | 0.0     | 0.05   | 322.648865     | 12.1665 
#3    | robust    | [512, 256, 128, 64] | 0.1    | 0.0     | 0.01   | 348.768555     | 12.1428 
#4    | robust    | [256, 128, 64]     | 0.1    | 0.0     | 0.01   | 351.149323     | 11.6752 
#5    | minmax    | [512, 256, 128, 64] | 0.0    | 0.1     | 0.005  | 368.512878     | 12.4263 




def create_neural_network(X_train, y_train):

    # 5. Architektur des Neuronalen Netzes definieren
    model = models.Sequential([
        # Eingabeschicht (Größe entspricht Anzahl der Features, z.B. 28)
        layers.Input(shape=(X_train.shape[1],)),
        layers.Dropout(0.1),

        layers.Dense(512, activation='relu'),#, kernel_regularizer=regularizers.l2(0.5)),
         # 1 versteckte Schicht
        layers.Dense(256, activation='relu'),#, kernel_regularizer=regularizers.l2(0.5)),
        layers.Dropout(0.1), # Verhindert Overfitting

        # 2 versteckte Schicht
        layers.Dense(128, activation='relu'),#, kernel_regularizer=regularizers.l2(0.5)),
        layers.Dropout(0.1), # Verhindert Overfitting
        
        # 3 versteckte Schicht
        layers.Dense(64, activation='relu'),#, kernel_regularizer=regularizers.l2(0.5)),
        layers.Dropout(0.1), # Verhindert Overfitting
        
        # Ausgabeschicht: 4 Neuronen für x, y, z, Gewicht (keine Aktivierungsfunktion für Regression)
        layers.Dense(4)
    ])
    
    # 6. Kompilieren
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.005)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    
    # 7. Training
    print("Starte Training...")

    # 3. Early Stopping
    early_stopping = EarlyStopping(
        monitor='val_loss', 
        patience=500, 
        restore_best_weights=True,
        verbose=0
    )
    history = model.fit(
        X_train_scaled, y_train,
        epochs=5000,
        batch_size=8,
        validation_split=0.1,
        callbacks=[early_stopping],
        verbose=0
    )

        # 5. Besten Validierungsfehler auslesen
    min_val_loss = min(history.history['val_loss'])
    corresponding_mae = history.history['val_mae'][history.history['val_loss'].index(min_val_loss)]
    epochs_trained = len(history.history['val_loss'])
    
    print(f" -> Fertig nach {epochs_trained} Epochen. Bester Val-Loss: {min_val_loss:.4f} mit MAE: {corresponding_mae:.4f}")
    
    return model

In [3]:
model = create_neural_network(X_train_scaled, y_train)

Starte Training...
 -> Fertig nach 763 Epochen. Bester Val-Loss: 76.3777 mit MAE: 6.3208


In [9]:
# scaler speichern
import joblib
joblib.dump(scaler, 'robust_scaler_Model_with_Weights_minmax_[512, 256, 128, 64]_LR_0.0_Drop_0.1_learn_0.005.pkl')
#Model Save:
model.save('Model_with_Weights_minmax_[512, 256, 128, 64]_LR_0.0_Drop_0.1_learn_0.005.keras')

In [4]:
import datetime
import re
import numpy as np
import pandas as pd
import tensorflow as tf  # Wichtig, um die Learning Rate sauber auszulesen

# --- Dein bestehender Code für die Vorhersage ---
y_pred = model.predict(X_test_scaled)

y_pred_df = pd.DataFrame(
    y_pred, columns=["x_pred", "y_pred", "z_pred","Gewicht_pred"], index=y_test.index
)

results = y_test.copy()
results[["x_pred", "y_pred", "z_pred","Gewicht_pred"]] = y_pred_df

# Fehler berechnen
results["abs_error_x"] = np.abs(results["x_pred"] - results["x"])
results["abs_error_y"] = np.abs(results["y_pred"] - results["y"])
results["abs_error_z"] = np.abs(results["z_pred"] - results["z"])
results["abs_error_Gewicht"] = np.abs(results["Gewicht_pred"] - results["Gewicht"])

results["mse_x"] = (results["x_pred"] - results["x"]) ** 2
results["mse_y"] = (results["y_pred"] - results["y"]) ** 2
results["mse_z"] = (results["z_pred"] - results["z"]) ** 2
results["mse_Gewicht"] = (results["Gewicht_pred"] - results["Gewicht"]) ** 2

# --- Restliche Spalten aus 'df' anhängen ---
#restliche_spalten = df.loc[y_test.index].drop(
#    columns=["x", "y", "z"], errors="ignore"
#)
#doppelte_spalten = [c for c in restliche_spalten.columns if c in results.columns]
#restliche_spalten = restliche_spalten.drop(columns=doppelte_spalten)

results = pd.concat([results, df.loc[y_test.index, ["source_file"]]], axis=1)


# --- NEU: Keras-Parameter dynamisch extrahieren ---

# 1. Zeitstempel erzeugen (Format: JYYMMDD_HHMMSS)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# 2. Modelltyp (wird "Sequential" sein)
model_name = model.__class__.__name__

try:
    # Netzarchitektur auslesen: Holt die Neuronenanzahl aller Dense-Schichten
    # Erzeugt eine Liste wie ['128', '64', '3']
    dense_units = [
        str(layer.units) for layer in model.layers if hasattr(layer, "units")
    ]
    arch_str = "-".join(dense_units)  # Wird zu "128-64-3"

    # Optimizer-Name auslesen (z.B. "Adam")
    opt_name = model.optimizer.__class__.__name__

    # Learning Rate auslesen (Keras speichert diese intern oft als Variable/Tensor)
    try:
        lr = float(tf.keras.backend.get_value(model.optimizer.learning_rate))
    except Exception:
        try:
            lr = float(model.optimizer.learning_rate.numpy())
        except Exception:
            lr = "unknown"

    # String für die Parameter zusammensetzen
    param_str = f"Arch_{arch_str}_{opt_name}_lr_{lr}"

except Exception:
    # Sicherer Fallback, falls die Keras-Version eine andere Struktur erzwingt
    param_str = "keras_nn"

# Dateiname zusammenbauen und ungültige Zeichen entfernen
filename = f"Auswertung_test_{model_name}_{param_str}_{timestamp}.csv"
filename = re.sub(r'[\\/*?:"<>| ]', "_", filename)

# Als CSV speichern (inklusive Index)
#results.to_csv(filename, index=True, sep=";", decimal=",")


# --- Konsolen-Ausgabe (unverändert bzw. erweitert) ---
print("Auswertung Testdatensatz:")
print(f"  - MAE x: {results['abs_error_x'].mean():.4f}")
print(f"  - MAE y: {results['abs_error_y'].mean():.4f}")
print(f"  - MAE z: {results['abs_error_z'].mean():.4f}")
print(f"  - MAE Gewicht: {results['abs_error_Gewicht'].mean():.4f}")
print(f"  - MSE x: {results['mse_x'].mean():.4f}")
print(f"  - MSE y: {results['mse_y'].mean():.4f}")
print(f"  - MSE z: {results['mse_z'].mean():.4f}")
print(f"  - MSE Gewicht: {results['mse_Gewicht'].mean():.4f}")

print(f"\n[INFO] Datei erfolgreich gespeichert unter: {filename}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Auswertung Testdatensatz:
  - MAE x: 13.7890
  - MAE y: 16.3487
  - MAE z: 5.5709
  - MAE Gewicht: 8.8593
  - MSE x: 374.4504
  - MSE y: 547.7657
  - MSE z: 58.0918
  - MSE Gewicht: 165.3442

[INFO] Datei erfolgreich gespeichert unter: Auswertung_test_Sequential_Arch_512-256-128-64-4_Adam_lr_0.004999999888241291_20260617_101849.csv


In [5]:
# Zeigt die ersten 20 Zeilen elegant an
results.style.format(
    {
        "x": "{:.2f}",
        "x_pred": "{:.2f}",
        "abs_error_x": "{:.2f}",
        "y": "{:.2f}",
        "y_pred": "{:.2f}",
        "abs_error_y": "{:.2f}",
        "z": "{:.2f}",
        "z_pred": "{:.2f}",
        "abs_error_z": "{:.2f}",
        "Gewicht": "{:.2f}",
        "Gewicht_pred": "{:.2f}",
        "abs_error_Gewicht": "{:.2f}",
    },
    decimal=",",  # Zeigt auch im Notebook Kommas statt Punkte!
).background_gradient(
    subset=["abs_error_x", "abs_error_y", "abs_error_z","abs_error_Gewicht"], cmap="Reds"
)

,x,y,z,Gewicht,x_pred,y_pred,z_pred,Gewicht_pred,abs_error_x,abs_error_y,abs_error_z,abs_error_Gewicht,mse_x,mse_y,mse_z,mse_Gewicht,source_file
103,"20,00","87,50","47,50","51,70","34,15","52,45","64,71","53,76","14,15","35,05","17,21","2,06","200,276633","1228,646585","296,225036","4,261828",Lumi_V_L_L_6.csv
139,"81,39","46,35","20,00","58,10","49,89","59,16","30,57","53,53","31,50","12,81","10,57","4,57","992,441107","164,051631","111,798250","20,896140",Endboss_L_U_S_X_3.csv
80,"47,50","87,50","20,00","51,70","76,06","47,74","21,35","47,71","28,56","39,76","1,35","3,99","815,675640","1580,681228","1,832335","15,892418",Lumi_L_L_8.csv
58,"25,00","50,00","31,75","127,80","33,90","36,28","45,08","125,70","8,90","13,72","13,33","2,10","79,217700","188,253193","177,733903","4,395733",Rechteck_V_L_L_2.csv
100,"20,00","87,50","47,50","51,70","67,77","38,95","37,82","52,22","47,77","48,55","9,68","0,52","2281,885840","2356,979823","93,621629","0,267608",Lumi_V_L_L_3.csv
30,"53,33","91,00","30,00","443,80","68,01","73,33","31,15","418,83","14,68","17,67","1,15","24,97","215,450498","312,221825","1,316724","623,289746",3eck_L_L_4.csv
107,"47,50","32,50","17,25","19,30","37,16","27,53","9,93","16,84","10,34","4,97","7,32","2,46","106,863938","24,733532","53,518484","6,035010",Durch_L_U_D_O_4.csv
84,"47,50","87,50","20,00","51,70","58,25","81,79","22,02","57,56","10,75","5,71","2,02","5,86","115,491485","32,613239","4,089199","34,357591",Lumi_L_L_12.csv
166,"12,50","120,00","17,50","71,60","67,53","36,36","24,37","93,89","55,03","83,64","6,87","22,29","3028,077412","6996,157453","47,196807","496,824346",Rechteck_Lang_L_L_3.csv
111,"47,50","32,50","5,75","19,30","51,17","46,00","12,79","28,54","3,67","13,50","7,04","9,24","13,436175","182,308919","49,527056","85,469004",Durch_L_U_D_U_2.csv


In [6]:
import datetime
import re
import numpy as np
import pandas as pd
import tensorflow as tf  # Wichtig, um die Learning Rate sauber auszulesen

# --- Dein bestehender Code für die Vorhersage ---
y_pred = model.predict(X_train_scaled)

y_pred_df = pd.DataFrame(
    y_pred, columns=["x_pred", "y_pred", "z_pred", "Gewicht_pred"], index=y_train.index
)

results = y_train.copy()
results[["x_pred", "y_pred", "z_pred", "Gewicht_pred"]] = y_pred_df

# Fehler berechnen
results["abs_error_x"] = np.abs(results["x_pred"] - results["x"])
results["abs_error_y"] = np.abs(results["y_pred"] - results["y"])
results["abs_error_z"] = np.abs(results["z_pred"] - results["z"])
results["abs_error_Gewicht"] = np.abs(results["Gewicht_pred"] - results["Gewicht"])

results["mse_x"] = (results["x_pred"] - results["x"]) ** 2
results["mse_y"] = (results["y_pred"] - results["y"]) ** 2
results["mse_z"] = (results["z_pred"] - results["z"]) ** 2
results["mse_Gewicht"] = (results["Gewicht_pred"] - results["Gewicht"]) ** 2

# --- Restliche Spalten aus 'df' anhängen ---
##restliche_spalten = df.loc[y_train.index].drop(
#    columns=["x", "y", "z"], errors="ignore"
#)
#doppelte_spalten = [c for c in restliche_spalten.columns if c in results.columns]
#restliche_spalten = restliche_spalten.drop(columns=doppelte_spalten)

#results = pd.concat([results, restliche_spalten], axis=1)
results = pd.concat([results, df.loc[y_train.index, ["source_file"]]], axis=1)

# --- NEU: Keras-Parameter dynamisch extrahieren ---

# 1. Zeitstempel erzeugen (Format: JYYMMDD_HHMMSS)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# 2. Modelltyp (wird "Sequential" sein)
model_name = model.__class__.__name__


try:
    # Netzarchitektur auslesen: Holt die Neuronenanzahl aller Dense-Schichten
    # Erzeugt eine Liste wie ['128', '64', '3']
    dense_units = [
        str(layer.units) for layer in model.layers if hasattr(layer, "units")
    ]
    arch_str = "-".join(dense_units)  # Wird zu "128-64-3"

    # Optimizer-Name auslesen (z.B. "Adam")
    opt_name = model.optimizer.__class__.__name__

    # Learning Rate auslesen (Keras speichert diese intern oft als Variable/Tensor)
    try:
        lr = float(tf.keras.backend.get_value(model.optimizer.learning_rate))
    except Exception:
        try:
            lr = float(model.optimizer.learning_rate.numpy())
        except Exception:
            lr = "unknown"

    # String für die Parameter zusammensetzen
    param_str = f"Arch_{arch_str}_{opt_name}_lr_{lr}"

except Exception:
    # Sicherer Fallback, falls die Keras-Version eine andere Struktur erzwingt
    param_str = "keras_nn"

# Dateiname zusammenbauen und ungültige Zeichen entfernen
filename = f"Auswertung_train_{model_name}_{param_str}_{timestamp}.csv"
filename = re.sub(r'[\\/*?:"<>| ]', "_", filename)

# Als CSV speichern (inklusive Index)
#results.to_csv(filename, index=True, sep=";", decimal=",")


# --- Konsolen-Ausgabe (unverändert bzw. erweitert) ---
print("Auswertung Trainingsdatensatz:")
print(f"  - MAE x: {results['abs_error_x'].mean():.4f}")
print(f"  - MAE y: {results['abs_error_y'].mean():.4f}")
print(f"  - MAE z: {results['abs_error_z'].mean():.4f}")
print(f"  - MAE Gewicht: {results['abs_error_Gewicht'].mean():.4f}")
print(f"  - MSE x: {results['mse_x'].mean():.4f}")
print(f"  - MSE y: {results['mse_y'].mean():.4f}")
print(f"  - MSE z: {results['mse_z'].mean():.4f}")
print(f"  - MSE Gewicht: {results['mse_Gewicht'].mean():.4f}")

print(f"\n[INFO] Datei erfolgreich gespeichert unter: {filename}")

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
Auswertung Trainingsdatensatz:
  - MAE x: 5.0140
  - MAE y: 6.2231
  - MAE z: 3.7385
  - MAE Gewicht: 5.8327
  - MSE x: 53.4804
  - MSE y: 80.7382
  - MSE z: 23.4800
  - MSE Gewicht: 80.1987

[INFO] Datei erfolgreich gespeichert unter: Auswertung_train_Sequential_Arch_512-256-128-64-4_Adam_lr_0.004999999888241291_20260617_101849.csv


In [7]:
# Zeigt die ersten 20 Zeilen elegant an
results.style.format(
    {
        "x": "{:.2f}",
        "x_pred": "{:.2f}",
        "abs_error_x": "{:.2f}",
        "y": "{:.2f}",
        "y_pred": "{:.2f}",
        "abs_error_y": "{:.2f}",
        "z": "{:.2f}",
        "z_pred": "{:.2f}",
        "abs_error_z": "{:.2f}",
        "Gewicht": "{:.2f}",
        "Gewicht_pred": "{:.2f}",
        "abs_error_Gewicht": "{:.2f}",
    },
    decimal=",",  # Zeigt auch im Notebook Kommas statt Punkte!
).background_gradient(
    subset=["abs_error_x", "abs_error_y", "abs_error_z","abs_error_Gewicht"], cmap="Reds"
)

,x,y,z,Gewicht,x_pred,y_pred,z_pred,Gewicht_pred,abs_error_x,abs_error_y,abs_error_z,abs_error_Gewicht,mse_x,mse_y,mse_z,mse_Gewicht,source_file
161,"120,00","12,50","17,50","71,60","113,16","14,41","22,26","70,02","6,84","1,91","4,76","1,58","46,848715","3,644843","22,647907","2,507634",Rechteck_Lang_L_U_4.csv
2,"50,00","50,00","50,00","185,50","44,23","49,76","47,79","168,82","5,77","0,24","2,21","16,68","33,344342","0,056989","4,905161","278,336700",4eck_g_O_V_u_3.csv
121,"54,50","47,50","20,00","43,90","53,92","55,47","20,40","46,78","0,58","7,97","0,40","2,88","0,335752","63,454658","0,160491","8,303976",6eck_L_U_6.csv
115,"47,50","32,50","5,75","19,30","51,22","34,61","9,57","20,28","3,72","2,11","3,82","0,98","13,801765","4,468151","14,570402","0,953669",Durch_L_U_D_U_6.csv
70,"87,50","47,50","20,00","51,70","90,49","50,06","22,65","53,57","2,99","2,56","2,65","1,87","8,963279","6,528066","6,996552","3,491651",Lumi_L_U_11.csv
136,"64,61","46,35","20,00","58,10","67,89","54,16","25,30","56,61","3,28","7,81","5,30","1,49","10,740236","61,034656","28,060987","2,212193",Endboss_L_U_S_0_3.csv
47,"31,75","50,00","25,00","127,80","27,86","47,62","27,17","122,41","3,89","2,38","2,17","5,39","15,153961","5,645373","4,691888","29,008158",Rechteck_L_L_7.csv
27,"53,33","91,00","30,00","443,80","66,36","71,55","30,42","408,29","13,03","19,45","0,42","35,51","169,826847","378,306833","0,173798","1260,948656",3eck_L_L_1.csv
145,"46,35","81,39","20,00","58,10","42,92","88,08","22,47","56,10","3,43","6,69","2,47","2,00","11,739516","44,768681","6,099286","4,019531",Endboss_L_L_S_Y_3.csv
86,"47,50","20,00","87,50","51,70","46,87","21,92","75,44","49,45","0,63","1,92","12,06","2,25","0,397762","3,674801","145,404163","5,050985",Lumi_H_L_U_1.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error

# Definition der 4 Drop-Varianten
drop_variants = {
    "No Drop": [],
    "Variant 1": [21, 43, 92, 159],
    "Variant 2": [92, 99, 31, 169, 150, 43],
    "Variant 3": [17, 21, 43, 53, 62, 92, 129, 150, 159, 166, 169],
    "Variant 4": [15, 17, 21, 30, 35, 43, 53, 62, 67, 72, 73, 75, 91, 92, 129, 139, 142, 150, 159, 166, 169]
}

# Definition der 5 Modell-Konfigurationen aus Zelle 8c0899a8
model_configs = [
    {"name": "Model 1", "scaler": "robust", "layers": [128, 64], "l2": 0.1, "dropout": 0.0, "lr": 0.05},
    {"name": "Model 2", "scaler": "robust", "layers": [512, 256, 128, 64], "l2": 0.5, "dropout": 0.0, "lr": 0.05},
    {"name": "Model 3", "scaler": "robust", "layers": [512, 256, 128, 64], "l2": 0.1, "dropout": 0.0, "lr": 0.01},
    {"name": "Model 4", "scaler": "robust", "layers": [256, 128, 64], "l2": 0.1, "dropout": 0.0, "lr": 0.01},
    {"name": "Model 5", "scaler": "minmax", "layers": [512, 256, 128, 64], "l2": 0.0, "dropout": 0.1, "lr": 0.005}
]

results_list = []
target_cols = ['x', 'y', 'z', 'Gewicht']

for drop_name, drop_indices in drop_variants.items():
    # Daten laden und vorbereiten
    df_full = pd.read_csv('training_data_ready.csv')
    # Nur existierende Indizes droppen
    indices_to_drop = [i for i in drop_indices if i in df_full.index]
    df_exp = df_full.drop(indices_to_drop, axis=0)
    
    x_cols = [col for col in df_exp.columns if col not in target_cols + ['source_file','Objektname','Objektkategorie','OAufnahme','Schwierigkeit','Gewicht']]
    
    X = df_exp[x_cols]
    y = df_exp[target_cols]
    
    # Train-Test-Split mit Validation Split bei Training (80/20 test -> 90/10 val in train)
    X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.1, random_state=42)
    
    for config in model_configs:
        print(f"Starte {drop_name} mit {config['name']}...")
        
        # Scaler wählen
        if config["scaler"] == "robust":
            scaler = RobustScaler()
        else:
            scaler = MinMaxScaler()
            
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)
        
        # Modell aufbauen
        model = models.Sequential()
        model.add(layers.Input(shape=(X_train_scaled.shape[1],)))
        
        for units in config["layers"]:
            model.add(layers.Dense(units, activation='relu', kernel_regularizer=regularizers.l2(config["l2"])))
            if config["dropout"] > 0:
                model.add(layers.Dropout(config["dropout"]))
                
        model.add(layers.Dense(4))
        
        optimizer = tf.keras.optimizers.Adam(learning_rate=config["lr"])
        model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
        
        early_stopping = EarlyStopping(monitor='val_loss', patience=30130, restore_best_weights=True, verbose=0)
        
        model.fit(
            X_train_scaled, y_train,
            epochs=1000,
            batch_size=8,
            validation_data=(X_val_scaled, y_val),
            callbacks=[early_stopping],
            verbose=0
        )
        
        # Evaluation
        preds_train = model.predict(X_train_scaled, verbose=0)
        preds_val = model.predict(X_val_scaled, verbose=0)
        preds_test = model.predict(X_test_scaled, verbose=0)
        
        mae_train = mean_absolute_error(y_train, preds_train, multioutput='raw_values')
        mae_val = mean_absolute_error(y_val, preds_val, multioutput='raw_values')
        mae_test = mean_absolute_error(y_test, preds_test, multioutput='raw_values')
        
        results_list.append({
            "Drop_Variant": drop_name,
            "Model_Config": config["name"],
            "Train_MAE_x": mae_train[0], "Train_MAE_y": mae_train[1], "Train_MAE_z": mae_train[2], "Train_MAE_Gewicht": mae_train[3],
            "Val_MAE_x": mae_val[0], "Val_MAE_y": mae_val[1], "Val_MAE_z": mae_val[2], "Val_MAE_Gewicht": mae_val[3],
            "Test_MAE_x": mae_test[0], "Test_MAE_y": mae_test[1], "Test_MAE_z": mae_test[2], "Test_MAE_Gewicht": mae_test[3]
        })

results_df = pd.DataFrame(results_list)
display(results_df)

Starte Variant 1 mit Model 1...
Starte Variant 1 mit Model 2...
Starte Variant 1 mit Model 3...
Starte Variant 1 mit Model 4...
Starte Variant 1 mit Model 5...
Starte Variant 2 mit Model 1...
Starte Variant 2 mit Model 2...
Starte Variant 2 mit Model 3...
Starte Variant 2 mit Model 4...
Starte Variant 2 mit Model 5...
Starte Variant 3 mit Model 1...
Starte Variant 3 mit Model 2...
Starte Variant 3 mit Model 3...
Starte Variant 3 mit Model 4...
Starte Variant 3 mit Model 5...
Starte Variant 4 mit Model 1...
Starte Variant 4 mit Model 2...
Starte Variant 4 mit Model 3...
Starte Variant 4 mit Model 4...
Starte Variant 4 mit Model 5...


,Drop_Variant,Model_Config,Train_MAE_x,Train_MAE_y,Train_MAE_z,Val_MAE_x,Val_MAE_y,Val_MAE_z,Test_MAE_x,Test_MAE_y,Test_MAE_z
0,Variant 1,Model 1,6.872856,5.064664,3.850614,11.253633,10.976868,6.647290,15.992828,17.790411,5.648474
1,Variant 1,Model 2,9.430860,10.574533,7.708986,12.669606,10.471148,8.333323,13.010736,13.799831,8.404533
2,Variant 1,Model 3,8.401100,7.005739,3.726586,10.653716,11.893679,5.657839,14.808399,19.510145,4.585637
3,Variant 1,Model 4,2.754220,2.432581,1.470721,12.398517,11.347514,4.908802,16.020546,19.290430,4.136741
4,Variant 1,Model 5,2.387145,2.250408,2.534258,11.789126,9.205834,5.357276,13.939017,15.459526,3.435226
5,Variant 2,Model 1,4.861551,5.413119,4.072461,8.941564,6.958927,5.652222,18.527706,15.882441,6.375751
6,Variant 2,Model 2,8.266910,9.758586,5.673196,10.188065,9.613363,6.196022,16.559505,18.860239,4.486477
7,Variant 2,Model 3,1.734056,2.540660,1.661490,9.005139,6.471330,5.433445,13.042131,15.887002,4.272820
8,Variant 2,Model 4,4.358511,2.707293,1.910377,9.665963,10.631804,3.540843,15.476354,14.491940,3.899290
9,Variant 2,Model 5,5.754449,6.042865,4.090389,9.500254,9.120817,4.303108,13.063194,14.106884,4.368253
